# Modelling NER

Ekspor pre-trained model `dslim/bert-base-NER` ke ONNX dan aplikasikan kuantisasi.

In [1]:
import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType
import numpy as np
import os

In [2]:
# Patch untuk menghindari WinError 32 PermissionError di Windows saat kuantisasi
import sys
import onnxruntime.quantization.quant_utils as quant_utils

def patched_load_model_with_shape_infer(model_path):
    import onnx
    from pathlib import Path
    inferred_model_path = quant_utils.generate_identified_filename(Path(model_path), '-inferred')
    onnx.shape_inference.infer_shapes_path(str(model_path), str(inferred_model_path))
    model = onnx.load(inferred_model_path.as_posix())
    quant_utils.add_infer_metadata(model)
    try:
        inferred_model_path.unlink()
    except PermissionError:
        pass # Abaikan error permission di Windows
    return model

quant_utils.load_model_with_shape_infer = patched_load_model_with_shape_infer
if 'onnxruntime.quantization.quantize' in sys.modules:
    sys.modules['onnxruntime.quantization.quantize'].load_model_with_shape_infer = patched_load_model_with_shape_infer

In [3]:
model_id = 'dslim/bert-base-NER'

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForTokenClassification.from_pretrained(model_id)
model.eval()

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [4]:
tokenizer.save_pretrained('./ner_tokenizer')

('./ner_tokenizer\\tokenizer_config.json',
 './ner_tokenizer\\special_tokens_map.json',
 './ner_tokenizer\\vocab.txt',
 './ner_tokenizer\\added_tokens.json',
 './ner_tokenizer\\tokenizer.json')

In [5]:
dummy_text = 'John Doe lives in New York and works as an Engineer.'
dummy_input = tokenizer(dummy_text, return_tensors='pt')
dummy_input

{'input_ids': tensor([[ 101, 1287, 2091, 1162, 2491, 1107, 1203, 1365, 1105, 1759, 1112, 1126,
         8252,  119,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [6]:
onnx_model_path = 'onnx_NER/ner_model.onnx'

torch.onnx.export(
    model,
    (dummy_input['input_ids'], dummy_input['attention_mask']),
    onnx_model_path,
    input_names=['input_ids', 'attention_mask'],
    output_names=['logits'],
    dynamic_axes={'input_ids': {0: 'batch_size', 1: 'sequence_length'},
                  'attention_mask': {0: 'batch_size', 1: 'sequence_length'},
                  'logits': {0: 'batch_size', 1: 'sequence_length'}},
    opset_version=14,
    do_constant_folding=True
)
print(f'Model exported to {onnx_model_path}')

d:\Hasil_Coding\Capstone_Project\model\.venv\lib\site-packages\transformers\modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask


Model exported to onnx_NER/ner_model.onnx


In [7]:
quantized_model_path = 'onnx_NER/ner_model_quantized.onnx'

quantize_dynamic(
    onnx_model_path,
    quantized_model_path,
    weight_type=QuantType.QUInt8
)
print(f'Quantized model exported to {quantized_model_path}')

Quantized model exported to onnx_NER/ner_model_quantized.onnx


In [8]:
session = ort.InferenceSession(quantized_model_path)
inputs = {
    'input_ids': dummy_input['input_ids'].numpy(),
    'attention_mask': dummy_input['attention_mask'].numpy()
}
outputs = session.run(None, inputs)

print('Logits shape:', outputs[0].shape)

predicted_ids = np.argmax(outputs[0], axis=-1)
print('Predicted tokens:', [tokenizer.decode(t) for t in predicted_ids[0]])

# Memetakan prediksi ke label entitas
labels = model.config.id2label
predicted_labels = [labels[id] for id in predicted_ids[0]]
print('Predicted labels:', predicted_labels)

Logits shape: (1, 15, 9)
Predicted tokens: ['[PAD]', '[unused3]', '[unused4]', '[unused4]', '[PAD]', '[PAD]', '[unused7]', '[unused8]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[unused1]', '[PAD]', '[PAD]']
Predicted labels: ['O', 'B-PER', 'I-PER', 'I-PER', 'O', 'O', 'B-LOC', 'I-LOC', 'O', 'O', 'O', 'O', 'B-MISC', 'O', 'O']
